In [2]:
#Echauffement : URL

num=3
typ="rue"
nom="tourelles"

f"https://api-adresse.data.gouv.fr/search/?q={"num"+typ+nom},Paris&limit=1"


'https://api-adresse.data.gouv.fr/search/?q=numruetourelles,Paris&limit=1'

In [3]:
#Echauffement : iloc et loc

import pandas as pd

df = pd.DataFrame({
    "number": [3, 18, 7, 12, 5],
    "name": ["tourelles", "BERNARDINS", "Bouvart", "Rivoli", "Seine"],
})
sub = df[df.number > 10]
print(sub)

#1) sub contient les rues dont le numéro est au moins égal à 11. Son index est [1,3], car seuls les données des lignes 1,3 subsistent.

#2) [NOTA : RETENIR : 
#  I- loc agit sur les index, qui peuvent ne pas être de la forme [0,N] après slicing, mais par exemple [2,4,5,11]. iloc agit sur les numéros de ligne, en commençant à compter à 0. 
# II- loc inclut la borne sup, iloc l'exclut.]

print(df.iloc[:2], df.loc[:2])

#Comme df n'a pas été slicé, son index est de la forme [0,N]. La seule différence est donc où loc et iloc vont-ils couper : loc garde une donnée de plus, car il inclut la borne sup.

#3) Comme l'index de loc est [1,3], sub.loc[:2] ne va garder que l'élément d'index 1 (car 3>2), tandis que sub.loc[:2] va tout garder, car sub contient 2 données, et iloc raisonne sur le nombre de lignes.

print(sub.iloc[:2], sub.loc[:2])

#4) sub.iloc[0] renvoie la première donnée de sub, tandis que sub.loc[0] renvoie une erreur : l'indexation débute à 1.

#5) 

#Avec iloc : 

print(sub.iloc[0])

#Avec loc : 

print(sub.loc[1]) #L'index du premier élément est 1

   number        name
1      18  BERNARDINS
3      12      Rivoli
   number        name
0       3   tourelles
1      18  BERNARDINS    number        name
0       3   tourelles
1      18  BERNARDINS
2       7     Bouvart
   number        name
1      18  BERNARDINS
3      12      Rivoli    number        name
1      18  BERNARDINS
number            18
name      BERNARDINS
Name: 1, dtype: object
number            18
name      BERNARDINS
Name: 1, dtype: object


In [4]:
#Géolocalisation

import pandas as pd

addresses = pd.read_csv('data/addresses.csv')
addresses.head(4)

addresses_small=addresses.iloc[:20]
addresses_small.to_csv(index=False)




'number,type,name,city\r\n5,rue,RIQUET,Paris\r\n141,boulevard,ITALIE,Paris\r\n234,rue,FAUBOURG ST-HONORÉ,Paris\r\n62,rue,MICHEL BIZOT,Paris\r\n6,rue,OURS,Paris\r\n30,rue,PETIT-MUSC,Paris\r\n3,impasse,MASSONET,Paris\r\n56,rue,PANOYAUX,Paris\r\n16,rue,THÉRÈSE,Paris\r\n11,rue,JEAN ROBERT,Paris\r\n22,rue,FAUBOURG POISSONNIÈRE,Paris\r\n19,rue,CHALIGNY,Paris\r\n77,rue,LILLE,Paris\r\n64,rue,BAGNOLET,Paris\r\n39,rue,ROMAINVILLE,Paris\r\n4,rue,PIERRE DUPONT,Paris\r\n3,place,MARCHÉ-Ste-CATHERINE,Paris\r\n19,rue,VIALA,Paris\r\n16,rue,ESPÉRANCE,Paris\r\n51,rue,GRENELLE,Paris\r\n'

In [5]:
#Une par une

#On observe des éléments avec des accolades, qui contiennent des éléments séparés par deux points. C'est certainement un dictionnaire.
#&limit=1 restreint au dictionnaire associé à la première adresse.

import requests
import time

t1=time.time() #Donne l'heure

def localize_one(num,typ,nom):
    r=requests.get(f"https://api-adresse.data.gouv.fr/search/?q={"num"+typ+nom},Paris&limit=1")
    if r.status_code>=300 or r.status_code<200 :
        return None
    else :
        return r.json()

localize_one(18,'rue','BENARDINS')

#On ne va pas raisonner ainsi, car le temps de se rendre sur le site du gouvernement pour une adresse rend le processus trop long pour des milliers d'addresses

t2=time.time()
temps_execution=t2-t1

print(temps_execution*20000)

#On obtient plus d'une heure pour 20 000 adresses, c'est bien trop. 




8363.361358642578


In [6]:
#En un seul coup

import requests
# this one is to fake a file from a string
import io

def mass_post(filename, col_number, col_type, col_name, col_city):

    # we don't care about the index for now
    # but note that if we want to merge the result
    # with the initial data, there is a need 
    # for a unique index...

    with open(filename) as feed:
        response = requests.post(
            "https://api-adresse.data.gouv.fr/search/csv/", 
            files={'data': feed},
            data={'columns': [col_number, col_type, col_name, col_city]})
    if not (200 <= response.status_code < 300):
        print(f"OOPS, got {response.status_code}")
        return None
    
    result = pd.read_csv(io.StringIO(response.text))

    return result


def localize_many(filename, col_number, col_type, col_name, col_city):
    geoloc=mass_post(filename, col_number, col_type, col_name, col_city)
    #mass_post est la localisation avec des colonnes inutiles à trier
    df=pd.read_csv(filename,encoding='utf-8') 
    #utf-8 est important pour s'affranchir des problèmes d'accent
    geoloc=geoloc[["latitude","longitude","result_city","result_type"]]
    return pd.merge(geoloc,df,left_index=True, right_index=True)
    #merge pour concaténer deux tableaux

# try your code on the small sample for starters

addresses_small_localized = localize_many("data/addresses-small.csv", "number", "type", "name", "city")
print(addresses_small_localized)

#Pour enlever les adresses non parisiennes à cause d'une ambiguïté

addresses_small=addresses_small.fillna(0)
addresses_small=addresses_small.loc[addresses_small['result_city']=='Paris']
print(addresses_small)

#On enregistre le fichier complet 

addresses=localize_many("data/addresses.csv","number","type","name","city")
addresses.to_csv(index=False)



    latitude  longitude       result_city  result_type  number       type  \
0  48.888134   2.376448             Paris  housenumber       5        rue   
1  46.669103  -1.418872  La Roche-sur-Yon       street     141  boulevard   
2  48.876160   2.302766             Paris  housenumber     234        rue   
3  48.835423   2.402040             Paris  housenumber      62        rue   
4  48.863311   2.351775             Paris  housenumber       6        rue   
5  48.852945   2.364722             Paris  housenumber      30        rue   
6  48.895498   2.351580             Paris  housenumber       3    impasse   
7  48.867023   2.388347             Paris  housenumber      56        rue   
8  48.866374   2.335212             Paris  housenumber      16        rue   
9  48.889520   2.358170             Paris  housenumber      11        rue   

                 name   city  
0              RIQUET  Paris  
1              ITALIE  Paris  
2  FAUBOURG ST-HONORÉ  Paris  
3        MICHEL BIZOT  Paris

KeyError: 'result_city'

In [15]:
#Afficher sur une carte

import folium

# Châtelet
CENTER = (48.856542, 2.347614)
   
m = folium.Map(location=CENTER, zoom_start=12)

def paris_map():
    return m

paris_map()


In [16]:
#On ajoute les adresses

def map_addresses(geoloc):
    for _, row in geoloc.iterrows():
        print(row)
        folium.Marker(location=(row['latitude'],row['longitude']),tooltip="Clique").add_to(m)
    
map_addresses(addresses_small_localized)
paris_map()

latitude         48.888134
longitude         2.376448
result_city          Paris
result_type    housenumber
number                   5
type                   rue
name                RIQUET
city                 Paris
Name: 0, dtype: object
latitude              46.669103
longitude             -1.418872
result_city    La Roche-sur-Yon
result_type              street
number                      141
type                  boulevard
name                     ITALIE
city                      Paris
Name: 1, dtype: object
latitude                 48.87616
longitude                2.302766
result_city                 Paris
result_type           housenumber
number                        234
type                          rue
name           FAUBOURG ST-HONORÉ
city                        Paris
Name: 2, dtype: object
latitude          48.835423
longitude           2.40204
result_city           Paris
result_type     housenumber
number                   62
type                    rue
name           MICH